In [14]:
import sys

! {sys.executable} -m pip install line_profiler

     |████████████████████████████████| 53 kB 453 kB/s eta 0:00:01
You should consider upgrading via the '/Users/bmac3/Code/ATAP/atap-roots-controls/venv/bin/python -m pip install --upgrade pip' command.


In this notebook we want to test the speed of different methods for reading in binary data

In [56]:
# setup for the notebook (importing libs and defining constants to be used)

from collections import namedtuple

import cbitstruct as bitstruct
from line_profiler import LineProfiler
import numpy as np
import pandas as pd

bin_file = "pixie16_binary_data.bin"

with open(bin_file, 'rb') as fp:
    bin_data = fp.read()
    
single_event_bin_data = bin_data[:16]
correct_bit_fields = pd.read_hdf('correct_bit_fields.hdf')

Let's make a list of the operation that need to be done to the data and think of ways we can combine/reorder/substitute the operations to make it faster.

1. get the data into memory
  * open a connection
    * read in all the bits at once
    * read in the bits in a loop
  * load the file directly into memory
    
2. parse the bits into their datatypes

3. combine the variables to get derived values

Let's get a sense of how long each individual operation will take.

In [47]:
# how long does it take to open a file?

def open_10_000():
    for _ in range(10_000):
        f = open(bin_file, "rb")


%timeit open_10_000()

198 ms ± 3.98 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


so 20.7 us for an open

In [48]:
# how long does it take to load in an entire file?

def load_bin_file():
    for _ in range(10_000):
        f = open(bin_file, "rb")
        r = f.read()

%timeit load_bin_file()

472 ms ± 15.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


so 49.1 us for open and read. This should translate to 49.1 - 20.7 = 28.4 us for read

In [49]:
# check if the previous conclusion is true with line_reader.

lp = LineProfiler(load_bin_file)
lp.run("load_bin_file()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.513689 s
File: <ipython-input-48-014c27ea96fb>
Function: load_bin_file at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def load_bin_file():
     4     10001       4934.0      0.5      1.0      for _ in range(10_000):
     5     10000     244174.0     24.4     47.5          f = open(bin_file, "rb")
     6     10000     264581.0     26.5     51.5          r = f.read()



26.5 us for read and 24.8 us for open. This looks about right.

In [58]:
# see how long this takes loading bin file into the local namespace first

def load_bin_file_local_namespace():
    global bin_file
    for _ in range(10_000):
        f = open(bin_file, "rb")
        r = f.read()

%timeit load_bin_file()

478 ms ± 16.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Even longer (no appreciable difference)

In [50]:
# see if using a context manager will change the runtime significantly

def load_bin_file_with_context_manager():
    for _ in range(10_000):
        with open(bin_file, "rb") as f:
            r = f.read()

%timeit load_bin_file_with_context_manager()

433 ms ± 9.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


so 43.3 us for open and read. looks like it takes slightly less time to use a context manager.

In [51]:
# check if the previous conclusion is true with line_reader

lp = LineProfiler(load_bin_file_with_context_manager)
lp.run("load_bin_file_with_context_manager()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.464032 s
File: <ipython-input-50-a7fe5da80d1f>
Function: load_bin_file_with_context_manager at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def load_bin_file_with_context_manager():
     4     10001       4836.0      0.5      1.0      for _ in range(10_000):
     5     10000     163555.0     16.4     35.2          with open(bin_file, "rb") as f:
     6     10000     295641.0     29.6     63.7              r = f.read()



Interestingly the opening operation looks like it takes less time with the context manager

We need to read in the loaded data as individual events. In order to do this we need a way of partitioning our data into data for each event. Let's assume we don't have traces and that each event is only the header.

In [41]:
# verify that our assumption is true for our sample data

print(correct_bit_fields['Event Length'].unique())
print(correct_bit_fields['Trace Length'].unique())

[4]
[0]


We see that the event length for every event is 4 so indeed we have only headers

In [60]:
# how long bitstruct will take for a single event?

HEADER_FORMATS = {
    0: (
        "b1u14u5u4u4u4",
        ("Finish Code", "Event Length", "Header Length", "CrateID", "SlotID", "Chan#"),
    ),
    1: ("u32", ("EVTTIME_LO",)),
    2: ("u3u13u16", ("CFD trigger source bits", "CFD Fractional Time", "EVTTIME_HI")),
    3: ("b1u15u16", ("Trace Out-of-Range Flag", "Trace Length", "Event Energy")),
}

bit_fmt = ""
field_names = ()

for i in HEADER_FORMATS.keys():
    bit_fmt += HEADER_FORMATS[i][0]
    field_names += HEADER_FORMATS[i][1]

def parse_binary_event_with_bitstruct():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = bitstruct.unpack(bit_fmt, header_bits)
        
        
lp = LineProfiler(parse_binary_event_with_bitstruct)
lp.run("parse_binary_event_with_bitstruct()")
lp.print_stats()

Timer unit: 1e-06 s

Total time: 0.023179 s
File: <ipython-input-60-e34f9f833786>
Function: parse_binary_event_with_bitstruct at line 20

Line #      Hits         Time  Per Hit   % Time  Line Contents
    20                                           def parse_binary_event_with_bitstruct():
    21     10001       3806.0      0.4     16.4      for _ in range(10_000):
    22     10000       7465.0      0.7     32.2          header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
    23     10000      11908.0      1.2     51.4          event_fields = bitstruct.unpack(bit_fmt, header_bits)



So it takes on average 31.8 + 51.1 = 82.9 us to parse an event this way with bitstruct.

In [65]:
str.replace(' ', ' ')

TypeError: replace expected at least 2 arguments, got 1

In [61]:
# see how long this takes if we want named tuples


Event = namedtuple('Event', field_names)


def parse_binary_event_with_bitstruct_and_namedtuples():
    for _ in range(10_000):
        header_bits = bitstruct.byteswap(4 * '4', single_event_bin_data)
        event_fields = bitstruct.unpack(bit_fmt, header_bits)
        event = Event(event_fields)
        
lp = LineProfiler(parse_binary_event_with_bitstruct_and_namedtuples)
lp.run("parse_binary_event_with_bitstruct_and_namedtuples()")
lp.print_stats()

ValueError: Type names and field names must be valid identifiers: 'Finish Code'